### Cell 1 — Verify Drive is mounted (same as B0, just confirm)

In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')

POS_DIR = "/content/drive/MyDrive/ArchP/binary_dataset/positive"
NEG_DIR = "/content/drive/MyDrive/ArchP/binary_dataset/negative"
VALID_EXT = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}

def count_images(folder_path):
    files = [f for f in os.listdir(folder_path)
             if os.path.splitext(f)[1].lower() in VALID_EXT]
    return len(files)

n_pos = count_images(POS_DIR)
n_neg = count_images(NEG_DIR)
print(f"Positive: {n_pos} | Negative: {n_neg} | Total: {n_pos + n_neg}")
print("Drive mounted OK")

Mounted at /content/drive
Positive: 530 | Negative: 2181 | Total: 2711
Drive mounted OK


### Cell 2 — Imports & Seeds

In [2]:
import os, random, numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB2
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from pathlib import Path

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TF: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


###  Cell 3 — Splits & Class Weights

In [4]:
POS_DIR = "/content/drive/MyDrive/ArchP/binary_dataset/positive"
NEG_DIR = "/content/drive/MyDrive/ArchP/binary_dataset/negative"
OUTPUT_DIR = "/content/outputs"
IMG_SIZE = 260      # ← B2 native resolution (B0 was 224)
BATCH_SIZE = 32

VALID_EXT = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}
os.makedirs(OUTPUT_DIR, exist_ok=True)

pos_files = [(str(p), 1) for p in sorted(Path(POS_DIR).iterdir())
             if p.suffix.lower() in VALID_EXT]
neg_files = [(str(p), 0) for p in sorted(Path(NEG_DIR).iterdir())
             if p.suffix.lower() in VALID_EXT]

df = pd.DataFrame(pos_files + neg_files, columns=['filepath', 'label'])

train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.176, stratify=train_val_df['label'], random_state=SEED)

train_df = train_df.copy(); train_df['split'] = 'train'
val_df   = val_df.copy();   val_df['split']   = 'val'
test_df  = test_df.copy();  test_df['split']  = 'test'

split_df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
split_df.to_csv(os.path.join(OUTPUT_DIR, "data_splits_b2.csv"), index=False)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_df['label'].values)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Class weights → Negative: {class_weight_dict[0]:.4f} | Positive: {class_weight_dict[1]:.4f}")

Train: 1898 | Val: 406 | Test: 407
Class weights → Negative: 0.6215 | Positive: 2.5580


### Cell 4 — Build Datasets

In [5]:
def load_and_preprocess(filepath, label):
    raw   = tf.io.read_file(filepath)
    image = tf.image.decode_image(raw, channels=3, expand_animations=False)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])   # 260×260 for B2
    image = tf.cast(image, tf.float32)                     # keep [0,255] — B2 normalizes internally
    return image, tf.cast(label, tf.float32)

def build_dataset(dataframe, shuffle=False):
    filepaths = dataframe['filepath'].values
    labels    = dataframe['label'].values.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED,
                        reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.ignore_errors()    # ← handles the corrupt PNG from last time
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = build_dataset(train_df, shuffle=True)
val_ds   = build_dataset(val_df,   shuffle=False)
test_ds  = build_dataset(test_df,  shuffle=False)

print("Datasets built — 260×260 px, ignore_errors active")
print(f"Train batches: {len(train_df) // BATCH_SIZE} | "
      f"Val batches: {len(val_df) // BATCH_SIZE} | "
      f"Test batches: {len(test_df) // BATCH_SIZE}")

Datasets built — 260×260 px, ignore_errors active
Train batches: 59 | Val batches: 12 | Test batches: 12


### Cell 5 — Build EfficientNet-B2

In [6]:
def build_efficientnet_b2(trainable_backbone=False):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_image")

    x = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(factor=0.25, fill_mode='reflect'),
        layers.RandomBrightness(factor=0.15),
        layers.RandomContrast(factor=0.15),
        layers.RandomZoom(height_factor=(-0.1, 0.1), fill_mode='reflect'),
    ], name="augmentation")(inputs)

    backbone = EfficientNetB2(
        include_top=False,
        weights='imagenet',
        input_tensor=x)
    backbone.trainable = trainable_backbone

    x = layers.GlobalAveragePooling2D(name="gap")(backbone.output)
    x = layers.BatchNormalization(name="head_bn")(x)
    x = layers.Dense(256, activation='relu', name="head_dense")(x)
    x = layers.Dropout(0.4, name="head_dropout")(x)
    outputs = layers.Dense(1, activation='sigmoid', name="output")(x)

    return keras.Model(inputs=inputs, outputs=outputs, name="EfficientNetB2_geoglyph")

model = build_efficientnet_b2(trainable_backbone=False)
model.summary(show_trainable=True)

31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "EfficientNetB2_geoglyph"

┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)      ┃ Output Shape    ┃   Param # ┃ Connected to   ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ input_image       │ (None, 260,     │         0 │ -              │   -   │
│ (InputLayer)      │ 260, 3)         │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ augmentation      │ (None, 260,     │         0 │ input_image[0… │   -   │
│ (Sequential)      │ 260, 3)         │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ rescaling         │ (None, 260,     │         0 │ augmentation[… │   -   │
│ (Rescaling)       │ 260, 3)         │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ normalization     │ (None, 260,     │         7 │ rescaling[0][… │   N   │
│ (Normalization)   │ 260, 3)         │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ rescaling_1       │ (None, 260,     │         0 │ normalization… │   -   │
│ (Rescaling)       │ 260, 3)         │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ stem_conv_pad     │ (None, 261,     │         0 │ rescaling_1[0… │   -   │
│ (ZeroPadding2D)   │ 261, 3)         │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ stem_conv         │ (None, 130,     │       864 │ stem_conv_pad… │   N   │
│ (Conv2D)          │ 130, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ stem_bn           │ (None, 130,     │       128 │ stem_conv[0][… │   N   │
│ (BatchNormalizat… │ 130, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ stem_activation   │ (None, 130,     │         0 │ stem_bn[0][0]  │   -   │
│ (Activation)      │ 130, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_dwconv    │ (None, 130,     │       288 │ stem_activati… │   N   │
│ (DepthwiseConv2D) │ 130, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_bn        │ (None, 130,     │       128 │ block1a_dwcon… │   N   │
│ (BatchNormalizat… │ 130, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_activati… │ (None, 130,     │         0 │ block1a_bn[0]… │   -   │
│ (Activation)      │ 130, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_se_squee… │ (None, 32)      │         0 │ block1a_activ… │   -   │
│ (GlobalAveragePo… │                 │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_se_resha… │ (None, 1, 1,    │         0 │ block1a_se_sq… │   -   │
│ (Reshape)         │ 32)             │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_se_reduce │ (None, 1, 1, 8) │       264 │ block1a_se_re… │   N   │
│ (Conv2D)          │                 │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_se_expand │ (None, 1, 1,    │       288 │ block1a_se_re… │   N   │
│ (Conv2D)          │ 32)             │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_se_excite │ (None, 130,     │         0 │ block1a_activ… │   - 

 Total params: 8,135,162 (31.03 MB)

 Trainable params: 363,777 (1.39 MB)

 Non-trainable params: 7,771,385 (29.65 MB)

### Cell 6 — Stage 1 Training

In [7]:
checkpoint_s1 = "/content/drive/MyDrive/ArchP/efficientnetb2_stage1_best.weights.h5"

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.BinaryAccuracy(name='accuracy'),
        keras.metrics.AUC(name='auc_roc', curve='ROC'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ]
)

checkpoint_cb = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_s1, monitor='val_auc_roc', mode='max',
    save_best_only=True, save_weights_only=True, verbose=1)

earlystop_cb = keras.callbacks.EarlyStopping(
    monitor='val_auc_roc', mode='max',
    patience=7, restore_best_weights=True, verbose=1)

reducelr_cb = keras.callbacks.ReduceLROnPlateau(
    monitor='val_auc_roc', mode='max',
    factor=0.5, patience=3, min_lr=1e-6, verbose=1)

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=[checkpoint_cb, earlystop_cb, reducelr_cb],
    verbose=1
)

Epoch 1/30
     60/Unknown 80s 1s/step - accuracy: 0.6138 - auc_roc: 0.6529 - loss: 0.8908 - precision: 0.2665 - recall: 0.6227

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 1: val_auc_roc improved from None to 0.62808, saving model to /content/drive/MyDrive/ArchP/efficientnetb2_stage1_best.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/ArchP/efficientnetb2_stage1_best.weights.h5
60/60 ━━━━━━━━━━━━━━━━━━━━ 98s 1s/step - accuracy: 0.6444 - auc_roc: 0.6869 - loss: 0.8913 - precision: 0.3056 - recall: 0.6442 - val_accuracy: 0.7309 - val_auc_roc: 0.6281 - val_loss: 0.5977 - val_precision: 0.3404 - val_recall: 0.4051 - learning_rate: 0.0010
Epoch 2/30
59/60 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step - accuracy: 0.6860 - auc_roc: 0.7610 - loss: 0.7721 - precision: 0.3507 - recall: 0.7251
Epoch 2: val_auc_roc did not improve from 0.62808
60/60 ━━━━━━━━━━━━━━━━━━━━ 21s 297ms/step - accuracy: 0.6865 - auc_roc: 0.7463 - loss: 0.7832 - precision: 0.3495 - recall: 0.7008 - val_accuracy: 0.7259 - val_auc_roc: 0.6255 - val_loss: 0.5839 - val_precision: 0.3182 - val_recall: 0.3544 - learning_rate: 0.0010
Epoch 3/30
59/60 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms

### Cell 7 — Stage 2 Training (BN freeze fix applied)

In [8]:
model.load_weights(checkpoint_s1)

# BN freeze fix — same critical pattern as B0
for layer in model.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

checkpoint_s2 = "/content/drive/MyDrive/ArchP/efficientnetb2_stage2_best.weights.h5"

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.BinaryAccuracy(name='accuracy'),
        keras.metrics.AUC(name='auc_roc', curve='ROC'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ]
)

checkpoint_cb_s2 = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_s2, monitor='val_auc_roc', mode='max',
    save_best_only=True, save_weights_only=True, verbose=1)

earlystop_cb_s2 = keras.callbacks.EarlyStopping(
    monitor='val_auc_roc', mode='max',
    patience=8, restore_best_weights=True, verbose=1)

reducelr_cb_s2 = keras.callbacks.ReduceLROnPlateau(
    monitor='val_auc_roc', mode='max',
    factor=0.5, patience=3, min_lr=1e-7, verbose=1)

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=[checkpoint_cb_s2, earlystop_cb_s2, reducelr_cb_s2],
    verbose=1
)

Epoch 1/30
     60/Unknown 32s 208ms/step - accuracy: 0.8664 - auc_roc: 0.9348 - loss: 0.4193 - precision: 0.5994 - recall: 0.8724
Epoch 1: val_auc_roc improved from None to 0.80455, saving model to /content/drive/MyDrive/ArchP/efficientnetb2_stage2_best.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/ArchP/efficientnetb2_stage2_best.weights.h5
60/60 ━━━━━━━━━━━━━━━━━━━━ 38s 313ms/step - accuracy: 0.8583 - auc_roc: 0.9329 - loss: 0.4271 - precision: 0.5948 - recall: 0.8625 - val_accuracy: 0.8321 - val_auc_roc: 0.8046 - val_loss: 0.4903 - val_precision: 0.5821 - val_recall: 0.4937 - learning_rate: 1.0000e-05
Epoch 2/30
59/60 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step - accuracy: 0.8312 - auc_roc: 0.9193 - loss: 0.4520 - precision: 0.5516 - recall: 0.8413
Epoch 2: val_auc_roc did not improve from 0.80455
60/60 ━━━━━━━━━━━━━━━━━━━━ 15s 248ms/step - accuracy: 0.8361 - auc_roc: 0.9166 - loss: 0.4505 - precision: 0.5534 - recall: 0.8383 - val_accuracy: 0.8420 - val_auc_roc: 0.79

The good news:

Epoch 1 started at 0.8046 — above Stage 1 best (0.7922) ✓ BN fix worked
0.8046 is already a solid number

The concern:

Stage 2 peaked at Epoch 1 and never improved. Early stopped at Epoch 9.
Train AUC kept climbing (0.93 → 0.96) while val stayed flat → the backbone is overfitting, not generalizing

B2 has 2× more parameters on the same 1,898 training images — fine-tuning hit the overfitting wall almost immediately. The test result will still be decent, but B2 may not beat B0 on test despite its larger capacity.

### Cell 8 — Official Test Evaluation:

In [9]:
# Load best Stage 2 weights
model.load_weights(checkpoint_s2)

# Evaluate on held-out test set
results = model.evaluate(test_ds, verbose=1)
metric_names = ['loss', 'accuracy', 'auc_roc', 'precision', 'recall']

print("\n===== EfficientNet-B2 Official Test Results =====")
for name, val in zip(metric_names, results):
    print(f"  {name:12s}: {val:.4f}")

# F1 Score
precision = results[3]
recall    = results[4]
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
print(f"  {'f1_score':12s}: {f1:.4f}")
print("=================================================")

13/13 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.8305 - auc_roc: 0.8235 - loss: 0.4747 - precision: 0.5696 - recall: 0.5625

===== EfficientNet-B2 Official Test Results =====
  loss        : 0.4747
  accuracy    : 0.8305
  auc_roc     : 0.8235
  precision   : 0.5696
  recall      : 0.5625
  f1_score    : 0.5660


Why B0 beat B2 — the core reason:
B2 has 2× more parameters (8.1M vs 4.4M) but the training set has only 371 positive samples. During Stage 2, B2's larger backbone had more capacity to memorize training patterns rather than learn generalizable geoglyph features — train AUC hit 0.95 while val stayed flat at 0.80. Classic overfitting.
B0's smaller, tighter architecture was actually the right fit for this dataset size. This is a genuinely interesting and publishable finding — compound scaling has diminishing returns when labeled data is the bottleneck, not model capacity.